## Concept 4 — Conversational Agent

### What is it?
A conversational agent maintains **full message history** across turns.
Every Human + AI + Tool message is appended to a history list and passed back
to the LLM on the next turn — so it can reference, correct, and build on prior context.

### Pipeline
```
Turn 1: [System] + [Human: Q1]                      → AI: A1
Turn 2: [System] + [Human: Q1, AI: A1, Human: Q2]  → AI: A2  (remembers Q1/A1)
Turn 3: [System] + [...all history..., Human: Q3]  → AI: A3  (remembers everything)
```

### Limitation (what Concept 5 fixes)
Limited to in-memory knowledge — cannot look up real documents dynamically.
→ Concept 5 wraps a vectorstore retriever as a tool so the agent fetches real docs.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

llm = get_llm()
llm_with_tools = llm.bind_tools(TOOLS)
tool_map = {t.name: t for t in TOOLS}

SYSTEM = 'You are a helpful assistant with access to tools. Remember and reference previous messages in the conversation.'
print('Conversational agent ready')

### Step 1 — Without Memory (Baseline)
Each call is independent — the LLM has no knowledge of prior turns.

In [ ]:
# No memory — two independent calls
r1 = llm.invoke('My favourite tool in LangChain is FAISS.')
print(f'Turn 1: {r1.content}')

r2 = llm.invoke('What is my favourite tool?')  # No history → can't know
print(f'Turn 2: {r2.content}')  # will say it doesn't know

### Step 2 — Build the Conversational Agent
Maintain a `history` list. Each turn appends Human + AI messages.
Tool calls are also appended so the LLM knows what was executed.

In [ ]:
history = [SystemMessage(content=SYSTEM)]

def chat(user_input: str) -> str:
    history.append(HumanMessage(content=user_input))

    response = llm_with_tools.invoke(history)
    history.append(response)

    # Execute any tool calls and add results to history
    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            print(f'  [Tool] {call["name"]}({call["args"]}) → {result}')
            history.append(ToolMessage(content=str(result), tool_call_id=call['id']))

        # LLM forms final answer using tool results
        final = llm_with_tools.invoke(history)
        history.append(AIMessage(content=final.content))
        return final.content

    return response.content

print(f'History initialized with {len(history)} system message(s)')

### Step 3 — Multi-Turn Conversation Demo
Watch the agent build on each answer across turns.

In [ ]:
turns = [
    'What is 144 divided by 12?',
    'Now multiply that result by 5.',           # references previous answer (12)
    'Search docs for: what is a LangChain agent?',
    'Summarize what we have done so far.',      # references all prior turns
]

for i, turn in enumerate(turns, 1):
    print(f'\nTurn {i} | Human: {turn}')
    answer = chat(turn)
    print(f'        AI:    {answer}')
    print(f'        [history length: {len(history)} messages]')

### Step 4 — Window Memory Strategy
For long sessions, pass only the last N messages to avoid growing context costs.

In [ ]:
WINDOW = 6  # keep system + last 6 messages (3 turns)
window_history = [SystemMessage(content=SYSTEM)]

def chat_windowed(user_input: str) -> str:
    window_history.append(HumanMessage(content=user_input))

    # Build context: system + last WINDOW messages
    context = [window_history[0]] + window_history[-WINDOW:]

    response = llm_with_tools.invoke(context)
    window_history.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            window_history.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        context2 = [window_history[0]] + window_history[-WINDOW:]
        final = llm_with_tools.invoke(context2)
        window_history.append(AIMessage(content=final.content))
        return final.content

    return response.content

windowed_turns = [
    'What is 25 multiplied by 4?',
    'What is the weather in Delhi?',
    'What two things did we just discuss?',
]

print('Windowed memory demo (window=6 messages):')
for t in windowed_turns:
    print(f'\nHuman: {t}')
    print(f'AI:    {chat_windowed(t)}')
    print(f'[window history: {len(window_history)} total, using last {min(WINDOW, len(window_history))}]')

### Full Function

In [ ]:
# Reset history for a clean session
session = [SystemMessage(content=SYSTEM)]

def conversational_agent(query: str) -> str:
    session.append(HumanMessage(content=query))
    response = llm_with_tools.invoke(session)
    session.append(response)
    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            session.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_tools.invoke(session)
        session.append(AIMessage(content=final.content))
        return final.content
    return response.content

# Demo
print('Session demo:')
from AgentUtils import TEST_QUERIES
for q in TEST_QUERIES[:3]:
    print(f'\nQ: {q}')
    print(f'A: {conversational_agent(q)}')
    print(f'[session: {len(session)} messages]')